# 02 · Student alignment and progressive pruning (DDP, 2 GPUs)
1. **Align (28 layers)**: the frozen student + LoRA r=16 on every decoder linear + a linear head 1024→256 learns to
   put its query vector on the teacher's vector of the same query: L2 = ||s − t||² = 2 − 2·cos. Only the adapter trains.
2. **Prune 28 → 14 → 7**: each stage merges the previous weights, keeps evenly spaced decoder layers (first and last
   always), then fine-tunes **every weight** + head with the same L2 on the same queries (re-alignment).

Each stage is its own checkpoint dir (`aligned/`, `prog_l14/`, `prog_l7/`, each with `final/`) and its own row.
Step checkpoints keep the last `keep_last`; re-running resumes and a finished stage is skipped.

Input: `01`. Output (attach to `03`, `04`): `runs/<mode>/{aligned, prog_l14, prog_l7}/`, `runs/<mode>/metrics.json`.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import os
import shutil
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = True    # use the same value as in 01
NPROCS = 2

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/notebooks/tensorpanda231"
IN_01 = f"{KAGGLE_IN}/hlm-01-teacher-embeddings" if ON_KAGGLE else REPO
DATA_PATH = f"{IN_01}/cache/{MODE}/data.json"
EMB_DIR = f"{IN_01}/cache/{MODE}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_01}/runs/{MODE}/metrics.json"

# --- stage 1: alignment of the 28-layer student (align.ALIGN_DEFAULTS; unknown keys raise)
ALIGN = dict(model="Qwen/Qwen3-Embedding-0.6B",   # student base, frozen
             dim=T["dim"],                        # head output = teacher index dim
             lora_rank=16, lora_alpha=32, lora_dropout=0.05,
             lora_targets=("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"),
             query_prefix="", max_query_len=32,   # student queries: no prompt
             loss="l2", loss_kwargs={},           # ||s - t||^2 = 2 - 2 cos between unit vectors
             lr=2e-4, head_lr=1e-3, weight_decay=0.0, warmup_ratio=0.05,
             epochs=1, batch_size=128,            # per GPU; global batch = batch_size * NPROCS
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=200, save_every=500, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR, out_dir=f"{RUN_DIR}/aligned")

# --- stages 2+: pruning, full fine-tune with L2 (prune.PRUNE_DEFAULTS; init / n_layers / out_dir set per stage)
STAGES = (14, 7)                                   # 28 -> 14 -> 7, each from the previous stage
PRUNE = dict(ffn_keep=1.0,                         # layers only: every kept layer keeps its full FFN
             dim=T["dim"], query_prefix="", max_query_len=32,
             lr=2e-5, head_lr=1e-4, weight_decay=0.0, warmup_ratio=0.05,
             epochs=1, batch_size=64,              # per GPU
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=200, save_every=500, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)

# --- evaluation (evaluation.evaluate_model kwargs): student queries vs the frozen teacher corpus, top 100
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, query_prefix="", device=get_device(), dtype="fp16")
LATENCY = dict(n_queries=2000, batch_sizes=(1, 32))

if DEBUG:
    for cfg in (ALIGN, PRUNE):
        cfg.update(log_every=5, eval_every=20, save_every=20)
    LATENCY.update(n_queries=200)

## 1 · Check inputs
Teacher vectors valid (finite, unit norm, `data.json` order) for the corpus, val/test and the alignment queries.

In [ ]:
from data import load_emb, load_json
from evaluation import check_emb, load_index

# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))   # (teacher docs, {split: teacher queries})
emb, ids = load_emb(EMB_DIR, "align")
check_emb(emb, ids, data["align"]["ids"], "align")
print(f"align {emb.shape}: {len(ids) // (ALIGN['batch_size'] * NPROCS)} align steps, "
      f"{len(ids) // (PRUNE['batch_size'] * NPROCS)} steps per prune stage")

## 2 · Align the 28-layer student
Rank r takes its slice of every global batch; DDP averages the adapter gradients. Val L2 / cos / nDCG@10 are logged
every `eval_every` steps.

In [ ]:
from align import align_main
from run_exp import train_once

runs = {"aligned": train_once(align_main, ALIGN, NPROCS)}

## 3 · Prune 28 → 14 → 7, re-align after each cut
`keep_layers` are indices into the original 28 layers.

In [ ]:
from data import load_json
from prune import prune_main

init = runs["aligned"]
for n in STAGES:
    cfg = {**PRUNE, "init": init, "n_layers": n, "out_dir": f"{RUN_DIR}/prog_l{n}"}
    init = runs[f"prog_l{n}"] = train_once(prune_main, cfg, NPROCS)
    print(f"prog_l{n} keeps layers", load_json(f"{init}/adapter_config.json")["keep_layers"])

## 4 · Training curves
`train_loss` and `val_loss` are both L2 (comparable across stages); a pruned stage starts higher and recovers.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from run_exp import plot_curves, save_fig

fig = plot_curves({n: f"{RUN_DIR}/{n}" for n in runs}, ("train_loss", "val_loss", "val_ndcg@10"), "align + prune")
display(save_fig(fig, RUN_DIR, "align_prune_curves"))
plt.close("all")

## 5 · Retrieval quality per stage (val + test)
Student queries vs the frozen teacher corpus. retention = nDCG@10 / teacher nDCG@10.

In [ ]:
from run_exp import evaluate_runs, print_rows

metrics = evaluate_runs(runs, data, index, RUN_DIR, EVAL)
for split in ("val", "test"):
    print_rows(metrics, list(runs), split)
    print()

## 6 · Query latency per stage (one GPU, fp16)

In [ ]:
from evaluation import measure_latency
from model import load_encoder
from teacher import free_gpu_memory

texts = data["val"]["texts"][:LATENCY["n_queries"]]
for name, path in runs.items():
    model, tok = load_encoder(path, EVAL["device"], torch.float16)
    print(f"--- {name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
    measure_latency(model, tok, texts, EVAL["device"], T["dim"], 32, "", LATENCY["batch_sizes"])
    del model
    free_gpu_memory()